# Notebook 03 — Test: can we use `gtab`?

**Goal:** before collecting all 20 keywords, test the tools with **one country (Viet Nam)** and **one keyword** (*điều dưỡng đức*).

`gtab` (West 2020) makes Google Trends values comparable. It builds an **anchor bank**: a ladder of reference searches from rare to very popular. Each keyword is then measured against the anchor of similar size. `gtab` downloads its data through the package `pytrends`.

**Risk:** Google sometimes blocks automatic downloads (error **429 – too many requests**). This notebook tests that first.

*Reference: West, R. (2020). Calibration of Google Trends Time Series. CIKM 2020.*

## Test 1 — Does `pytrends` work? (one request, a few seconds)

In [ ]:
from pytrends.request import TrendReq

pytrends = TrendReq(hl="en-US", tz=0)
pytrends.build_payload(["điều dưỡng đức"], timeframe="2015-01-01 2026-10-31", geo="VN")
test = pytrends.interest_over_time()

print("Rows:", len(test))
test.tail()

**Check:** the values should be similar to the manual download `data/raw/gt_keywords/round2/VN_care.csv` (not exactly the same, because Google uses a new sample each time).

- ✅ It works → go on to Test 2.
- ❌ Error `429` → Google blocks us. Wait 1–2 hours and try once more. If it fails again, we use plan B (manual downloads).
- ❌ Other error → send a screenshot.

## Test 2 — Build one anchor bank with `gtab` (slow: 10–40 minutes)

The anchor bank is built **once per country and period** and saved in `data/gtab/`. It needs many requests to Google, so it is slow. Do not stop the cell while it runs.

In [ ]:
%pip install "urllib3<2"

In [5]:
import importlib, inspect
import gtab, gtab.core

# Force Python to load the fixed gtab file 
importlib.reload(gtab.core)
importlib.reload(gtab)
print("Fixed version loaded:", "PATCH" in inspect.getsource(gtab.core))

t = gtab.GTAB(dir_path="../data/gtab_workspace")
t.set_options(pytrends_config={"geo": "VN", "timeframe": "2015-01-01 2026-10-31"})
t.create_anchorbank()                          # slow: many Google requests

Fixed version loaded: True
Directory already exists, loading data from it.
Using directory '../data/gtab_workspace'
Active anchorbank changed to: google_anchorbank_geo=_timeframe=2019-01-01 2020-08-01.tsv

Start AnchorBank init for region VN in timeframe 2015-01-01 2026-10-31...
GTAB with such parameters already exists! Load it with 'set_active_gtab(filename)' or rename/delete it to create another one with this name.


## Test 3 — Calibrate one keyword

In [6]:
t.list_gtabs()                                   # shows the saved anchor banks

Existing GTABs:
	google_anchorbank_geo=AR_timeframe=2019-01-01 2020-12-31.tsv
	google_anchorbank_geo=AU_timeframe=2019-01-01 2020-12-31.tsv
	google_anchorbank_geo=BA_timeframe=2019-01-01 2020-12-31.tsv
	google_anchorbank_geo=BO_timeframe=2019-01-01 2020-12-31.tsv
	google_anchorbank_geo=BR_timeframe=2019-01-01 2020-12-31.tsv
	google_anchorbank_geo=CA_timeframe=2019-01-01 2020-12-31.tsv
	google_anchorbank_geo=CN_timeframe=2019-01-01 2020-12-31.tsv
	google_anchorbank_geo=CO_timeframe=2019-01-01 2020-12-31.tsv
	google_anchorbank_geo=DE_timeframe=2017-03-05 2021-05-05.tsv
	google_anchorbank_geo=DE_timeframe=2018-03-05 2021-05-05_cat=71.tsv
	google_anchorbank_geo=DE_timeframe=2019-01-01 2020-12-31.tsv
	google_anchorbank_geo=DE_timeframe=2020-03-05 2021-05-05_cat=71.tsv
	google_anchorbank_geo=DK_timeframe=2019-01-01 2020-12-31.tsv
	google_anchorbank_geo=EC_timeframe=2019-01-01 2020-12-31.tsv
	google_anchorbank_geo=ES_timeframe=2019-01-01 2020-12-31.tsv
	google_anchorbank_geo=FR_timeframe=2019

In [7]:
# Switch to the Viet Nam anchor bank (gtab does not do this automatically)
t.set_active_gtab("google_anchorbank_geo=VN_timeframe=2015-01-01 2026-10-31.tsv")

# Calibrate one keyword (a few Google requests)
result = t.new_query("điều dưỡng đức")
result.head()

Active anchorbank changed to: google_anchorbank_geo=VN_timeframe=2015-01-01 2026-10-31.tsv

Using ../data/gtab_workspace\output\google_anchorbanks\google_anchorbank_geo=VN_timeframe=2015-01-01 2026-10-31.tsv
New query 'điều dưỡng đức'
New query calibrated!


,max_ratio,max_ratio_hi,max_ratio_lo
date,,,
2015-01-01,0.231579,0.238095,0.225131
2015-02-01,0.263158,0.269841,0.256545
2015-03-01,0.294737,0.301587,0.287958
2015-04-01,0.378947,0.386243,0.371728
2015-05-01,0.305263,0.312169,0.298429


**What the result means**
- `max_ratio`: the calibrated value — on a **common scale** with all other keywords in Viet Nam
- `max_ratio_hi`, `max_ratio_lo`: upper and lower error bounds

If all three tests work, the method works and we can do all 20 keywords (Notebook 04).